# Exercise 1 — PaperAccount: buy and portfolio_value

`PaperAccount` is the core state machine. It tracks cash and shares separately so every dollar is accounted for. `portfolio_value` is the mark-to-market total at the current price. `buy` converts cash into shares using a `fraction` of available cash.

In [ ]:
import pandas as pd, math
from dataclasses import dataclass, field

def _synthetic(n=252):
    prices = [100.0 * (1 + 0.3 * math.sin(i * 2 * math.pi / n)) for i in range(n)]
    dates  = pd.date_range("2023-01-01", periods=n, freq="B")
    close  = pd.Series(prices, index=dates)
    return pd.DataFrame({
        "Open":   close.shift(1).fillna(close.iloc[0]),
        "High":   close * 1.01,
        "Low":    close * 0.99,
        "Close":  close,
        "Volume": pd.Series([1_000_000 + i * 1_000 for i in range(n)], index=dates),
    })
@dataclass
class Trade:
    date:        object
    action:      str
    price:       float
    shares:      float
    cash_after:  float
    value_after: float

@dataclass
class PaperAccount:
    initial_cash: float = 10_000.0
    cash:         float = field(init=False)
    shares:       float = field(init=False)
    trades:       list  = field(init=False)

    def __post_init__(self):
        # TODO: self.cash = initial_cash, shares = 0.0, trades = []
        self.cash   = self.initial_cash
        self.shares = 0.0
        self.trades = []

    def portfolio_value(self, price):
        """Cash + shares * price."""
        # TODO: one line
        return 0.0

    def buy(self, date, price, fraction=1.0):
        """Buy fraction of cash worth of shares.

        Steps:
          1. Guard: if cash<=0 or price<=0, return None
          2. shares = (self.cash * fraction) / price
          3. cost   = shares * price
          4. Guard: if cost > self.cash: recalculate to avoid float rounding
          5. self.cash -= cost; self.shares += shares
          6. Append Trade(...) to self.trades
          7. Return the Trade
        """
        # TODO: ~8 lines
        return None

    def sell(self, date, price):
        """Placeholder — implement in Exercise 2."""
        return None


### Checks

In [ ]:
checks = 0

# 1 — __post_init__: correct initial state
try:
    acc = PaperAccount(initial_cash=10_000.0)
    assert abs(acc.cash   - 10_000.0) < 1e-9
    assert abs(acc.shares - 0.0)      < 1e-9
    assert acc.trades == []
    checks += 1; print("✅ 1 PaperAccount initial state: cash=10000, shares=0, trades=[]")
except Exception as e:
    print("❌ 1:", e)

# 2 — portfolio_value: cash + shares * price
try:
    acc = PaperAccount(10_000.0)
    acc.cash   = 5_000.0
    acc.shares = 25.0
    assert abs(acc.portfolio_value(100.0) - 7_500.0) < 1e-9,         f"expected 7500, got {acc.portfolio_value(100.0)}"
    checks += 1; print("✅ 2 portfolio_value = cash + shares × price = 7500")
except Exception as e:
    print("❌ 2:", e)

# 3 — buy: reduces cash, increases shares
try:
    acc = PaperAccount(10_000.0)
    t   = acc.buy("2023-01-01", 100.0)
    assert t is not None and t.action == "BUY"
    assert abs(acc.shares - 100.0) < 1e-6, f"expected 100 shares, got {acc.shares}"
    assert abs(acc.cash)           < 1e-6, f"expected 0 cash, got {acc.cash}"
    checks += 1; print("✅ 3 buy(price=100, cash=10000) → 100 shares, 0 cash")
except Exception as e:
    print("❌ 3:", e)

# 4 — portfolio_value unchanged after buy (price same as buy price)
try:
    acc = PaperAccount(10_000.0)
    acc.buy("2023-01-01", 100.0)
    pv = acc.portfolio_value(100.0)
    assert abs(pv - 10_000.0) < 1e-6,         f"portfolio_value should be 10000 right after buy, got {pv}"
    checks += 1; print("✅ 4 portfolio_value equals initial_cash immediately after buy")
except Exception as e:
    print("❌ 4:", e)

# 5 — buy with fraction=0.5: half invested, half cash remains
try:
    acc = PaperAccount(10_000.0)
    t   = acc.buy("2023-01-01", 100.0, fraction=0.5)
    assert abs(acc.shares - 50.0)   < 1e-6, f"expected 50 shares, got {acc.shares}"
    assert abs(acc.cash - 5_000.0)  < 1e-6, f"expected 5000 cash, got {acc.cash}"
    checks += 1; print("✅ 5 buy(fraction=0.5) → 50 shares, $5000 cash remaining")
except Exception as e:
    print("❌ 5:", e)

print(f"\n{checks}/5 checks passed!")
